In [10]:
from pathlib import Path
import json

import numpy as np

from src.training.train import load_train_arrays, train_tabular_model, save_trained_pipeline

In [11]:
MAIN_DIR = Path("/home/dani/github/profecia/data")

SPLITS_BASE_DIR = MAIN_DIR / "splits"
MODELS_BASE_DIR = MAIN_DIR / "models"

processed_run_name = "land_ebf_bs_annual"
split_mask_name = "landcover"      # "climate" o "landcover"

model_name = "rf"                  # "rf", "hgb", "mlp"
model_run_name = "rf_baseline"
scaler_name = None                 # None, "standard", "robust"

random_state = 42
mmap_mode = "r"

INPUT_DIR = SPLITS_BASE_DIR / processed_run_name / split_mask_name
MODEL_DIR = MODELS_BASE_DIR / processed_run_name / split_mask_name

print("INPUT_DIR:", INPUT_DIR)
print("MODEL_DIR:", MODEL_DIR)
print("model_name:", model_name)
print("model_run_name:", model_run_name)
print("scaler_name:", scaler_name)

INPUT_DIR: /home/dani/github/profecia/data/splits/land_ebf_bs_annual/landcover
MODEL_DIR: /home/dani/github/profecia/data/models/land_ebf_bs_annual/landcover
model_name: rf
model_run_name: rf_baseline
scaler_name: None


In [12]:
RF_PARAMS = {
    "n_estimators": 50,
    "max_depth": 20,
    "min_samples_leaf": 5,
    "max_features": "sqrt",
    "n_jobs": 1,
    "bootstrap": True,
    "max_samples": 0.3,
}

HGB_PARAMS = {
    "learning_rate": 0.1,
    "max_iter": 200,
    "max_depth": None,
    "min_samples_leaf": 20,
}

MLP_PARAMS = {
    "hidden_layer_sizes": (128, 64),
    "activation": "relu",
    "solver": "adam",
    "alpha": 1e-4,
    "learning_rate": "adaptive",
    "learning_rate_init": 1e-3,
    "max_iter": 300,
    "early_stopping": True,
    "validation_fraction": 0.1,
}

In [13]:
if model_name == "rf":
    model_params = RF_PARAMS
elif model_name == "hgb":
    model_params = HGB_PARAMS
elif model_name == "mlp":
    model_params = MLP_PARAMS
else:
    raise ValueError("model_name debe ser 'rf', 'hgb' o 'mlp'")

print("model_params:", model_params)

model_params: {'n_estimators': 50, 'max_depth': 20, 'min_samples_leaf': 5, 'max_features': 'sqrt', 'n_jobs': 1, 'bootstrap': True, 'max_samples': 0.3}


## Cargar arrays de train

In [14]:
X_train, y_train, dataset_metadata = load_train_arrays(
    input_dir=INPUT_DIR,
    mmap_mode=mmap_mode,
)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print(json.dumps(dataset_metadata, indent=2, ensure_ascii=False))

X_train shape: (538412, 6)
y_train shape: (538412,)
{
  "prefix": "land_ebf_bs_annual_landcover_30pct_pixel_fraction",
  "n_train": 538412,
  "n_test": 59819,
  "n_features": 6,
  "feature_names": [
    "SM1",
    "SM2",
    "TP",
    "T2M",
    "SSRD",
    "VPD"
  ],
  "target": "LAI",
  "target_dtype": "float32",
  "feature_dtypes": {
    "SM1": "float32",
    "SM2": "float32",
    "TP": "float32",
    "T2M": "float32",
    "SSRD": "float32",
    "VPD": "float32"
  },
  "n_time": 41,
  "time_start": "1982-01-01 00:00:00",
  "time_end": "2022-01-01 00:00:00",
  "time_values": [
    "1982-01-01 00:00:00",
    "1983-01-01 00:00:00",
    "1984-01-01 00:00:00",
    "1985-01-01 00:00:00",
    "1986-01-01 00:00:00",
    "1987-01-01 00:00:00",
    "1988-01-01 00:00:00",
    "1989-01-01 00:00:00",
    "1990-01-01 00:00:00",
    "1991-01-01 00:00:00",
    "1992-01-01 00:00:00",
    "1993-01-01 00:00:00",
    "1994-01-01 00:00:00",
    "1995-01-01 00:00:00",
    "1996-01-01 00:00:00",
    "1997

In [15]:
assert X_train.ndim == 2
assert y_train.ndim == 1
assert X_train.shape[0] == y_train.shape[0]
assert X_train.shape[1] == dataset_metadata["n_features"]

print("NaN en X_train:", np.isnan(X_train).sum())
print("NaN en y_train:", np.isnan(y_train).sum())

assert np.isnan(X_train).sum() == 0
assert np.isnan(y_train).sum() == 0

print("No hay NaN")

NaN en X_train: 0
NaN en y_train: 0
No hay NaN


In [16]:
print("y_train min :", float(np.min(y_train)))
print("y_train max :", float(np.max(y_train)))
print("y_train mean:", float(np.mean(y_train)))
print("y_train std :", float(np.std(y_train)))

y_train min : 0.020370369777083397
y_train max : 5.518518447875977
y_train mean: 1.1263179779052734
y_train std : 0.8757663369178772


## Entrenar modelo

In [17]:
train_result = train_tabular_model(
    X_train=X_train,
    y_train=y_train,
    model_name=model_name,
    scaler_name=scaler_name,
    random_state=random_state,
    **model_params,
)

print("Modelo entrenado:", train_result["train_info"]["model_name"])
print("Scaler:", train_result["train_info"]["scaler_name"])
train_result["train_info"]

Modelo entrenado: RandomForestRegressor
Scaler: none


{'model_name_requested': 'rf',
 'model_name': 'RandomForestRegressor',
 'model_params': {'bootstrap': True,
  'ccp_alpha': 0.0,
  'criterion': 'squared_error',
  'max_depth': 20,
  'max_features': 'sqrt',
  'max_leaf_nodes': None,
  'max_samples': 0.3,
  'min_impurity_decrease': 0.0,
  'min_samples_leaf': 5,
  'min_samples_split': 2,
  'min_weight_fraction_leaf': 0.0,
  'monotonic_cst': None,
  'n_estimators': 50,
  'n_jobs': 1,
  'oob_score': False,
  'random_state': 42,
  'verbose': 0,
  'warm_start': False},
 'scaler_name': 'none',
 'random_state': 42,
 'n_rows_train': 538412,
 'n_features': 6,
 'X_dtype': 'float32',
 'y_dtype': 'float32'}

In [18]:
saved_paths = save_trained_pipeline(
    output_dir=MODEL_DIR,
    model=train_result["model"],
    scaler=train_result["scaler"],
    train_info=train_result["train_info"],
    dataset_metadata=dataset_metadata,
    prefix=model_run_name,
)

saved_paths

{'model_path': '/home/dani/github/profecia/data/models/land_ebf_bs_annual/landcover/rf_baseline_model.joblib',
 'scaler_path': None,
 'train_info_path': '/home/dani/github/profecia/data/models/land_ebf_bs_annual/landcover/rf_baseline_train_info.json'}